<a href="https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy python-dotenv

import os
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]
daily_files = [
    hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                     filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
    for m in MONTHS
]
dim_content_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                    filename="dim_content.parquet", token=token)
clients_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                filename="dim_clients.parquet", token=token)

con = duckdb.connect()
file_list = ", ".join(f"'{f}'" for f in daily_files)
REL = f"read_parquet([{file_list}])"
DECISION_DATE = "2026-03-31"

query = f"""
WITH prior AS (
    SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions, SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_sum_position) AS gsc_sum_position,
        -- Reproduces the discarded gate (`gsc_avg_position > 0`) so the next
        -- cell can measure what it used to throw away. Not a feature.
        SUM(gsc_impressions) FILTER (WHERE gsc_avg_position > 0) AS old_gate_impressions,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
        COUNT(*) FILTER (WHERE ga4_sessions > 0) AS days_with_sessions,
        BOOL_OR(ga4_data_available) AS ga4_data_available,
        SUM(ga4_pageviews) AS ga4_pageviews, SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_users) AS ga4_users, SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,
        SUM(sessions_organic) AS sessions_organic, SUM(sessions_direct) AS sessions_direct,
        SUM(scroll_events) AS scroll_events,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 60 DAY
              AND report_date < DATE '{DECISION_DATE}' - INTERVAL 30 DAY
        ) AS trend_baseline_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 30 DAY
        ) AS trend_recent_impr
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
      AND report_date < DATE '{DECISION_DATE}'
    GROUP BY content_hash_id
),
future AS (
   
    SELECT content_hash_id, SUM(gsc_impressions) AS future_impressions
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}'
      AND report_date < DATE '{DECISION_DATE}' + INTERVAL 30 DAY
    GROUP BY content_hash_id
)
SELECT p.*, f.future_impressions
FROM prior p JOIN future f USING (content_hash_id)
WHERE p.trend_baseline_impr > 0 AND p.trend_recent_impr > 0
-- Deterministic row order. Without it DuckDB may return rows in any order, so
-- the train/test split shifts between runs even with a fixed seed -- results
-- stop being reproducible from a fresh clone.
ORDER BY content_hash_id
"""
df = con.sql(query).df()

# Google's documented formula for GSC bulk-export data (https://support.google.com/webmasters/answer/12917991):
#   average position (1-based) = SUM(sum_position) / SUM(impressions) + 1
# sum_position is ZERO-based -- 0 is the top result. No filtering is needed:
# zero-impression rows carry sum_position = 0, so they contribute nothing to
# either side (verified: 0 rows violate this).
df["gsc_avg_position"] = df["gsc_sum_position"] / df["gsc_impressions"] + 1
assert df["gsc_avg_position"].notna().all(), "cohort should guarantee prior-window impressions"

# dim_content carries its own client_hash_id; dropping it avoids a column
# collision that silently breaks the dim_clients merge below.
dim = con.sql(f"SELECT * FROM read_parquet('{dim_content_file}')").df()
dim = dim.drop(columns=["client_hash_id"])
df = df.merge(dim, on="content_hash_id", how="left")

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM read_parquet('{clients_file}')").df()
df = df.merge(clients, on="client_hash_id", how="left")

print("Rows before any filtering:", len(df))

# NOTE: an is_deleted / is_published filter was applied here and has been removed.
# dim_content is an export-time snapshot (July 2026), so those flags describe a
# page's status months AFTER this decision point. A page deleted in June was live
# in March and legitimately belongs in a March population; excluding it is
# selection on the outcome window. ML-06 section 1 has the evidence.

prior_window_start = pd.Timestamp(DECISION_DATE) - pd.Timedelta(days=90)
coverage_ok = df["gsc_data_start"].isna() | (df["gsc_data_start"] <= prior_window_start)
print("Dropped for incomplete client coverage:", (~coverage_ok).sum(),
      f"({(~coverage_ok).mean():.1%})")
df = df[coverage_ok].copy()
print("Rows after coverage filter:", len(df))
print()

df["prior_trend_pct"] = (df["trend_recent_impr"] - df["trend_baseline_impr"]) / df["trend_baseline_impr"] * 100
df["was_declining"] = df["prior_trend_pct"] <= -20

decision_ts = pd.Timestamp(DECISION_DATE)
df["content_age_days"] = (decision_ts - pd.to_datetime(df["content_created_date"])).dt.days
# Retained as evidence, not a feature: its negative values are what exposed
# dim_content as an export-time snapshot.
df["days_since_last_update"] = (decision_ts - pd.to_datetime(df["content_updated_date"])).dt.days

# Point-in-time reconstruction. dim_content stores only the LATEST update, so:
#   - update visible on or before the decision point -> that IS the last update. Exact.
#   - update after the decision point -> no update is visible before D, so fall back
#     to creation ("last touched at creation as far as D could know").
# Known flaw: a page updated BOTH before and after D shows only the later date, so the
# fallback overstates staleness for it -- and those errors are not random, since pages
# updated after D are plausibly ones already identified as needing work.
upd = pd.to_datetime(df["content_updated_date"])
crt = pd.to_datetime(df["content_created_date"])
visible = upd <= decision_ts
df["update_visible_before_d"] = visible.astype(int)
df["days_since_update_pit"] = (decision_ts - upd.where(visible, crt)).dt.days

df["ctr"] = (df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan) * 100).fillna(0)
df["engagement_rate"] = (df["ga4_engaged_sessions"] / df["ga4_sessions"].replace(0, np.nan) * 100).fillna(0)
df["scroll_rate"] = (df["scroll_events"] / df["ga4_pageviews"].replace(0, np.nan) * 100).fillna(0)

# Flags must be computed BEFORE the fills below, or the missingness is erased.
df["has_ga4_data"] = df["ga4_data_available"].fillna(False).astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_backlink_data"] = df["backlinks"].notna().astype(int)

for col in ["search_volume", "competition", "cpc", "word_count", "char_count", "backlinks",
            "ga4_pageviews", "ga4_sessions", "ga4_users", "ga4_engaged_sessions",
            "ga4_total_engagement_sec", "sessions_organic", "sessions_direct", "scroll_events"]:
    df[col] = df[col].fillna(0)
df["main_intent"] = df["main_intent"].fillna("unknown")
df["content_type"] = df["content_type"].fillna("unknown")
df["competition_level"] = df["competition_level"].fillna("unknown")

for col in ["gsc_impressions", "gsc_clicks", "ga4_sessions", "search_volume", "backlinks",
            "scroll_events", "gsc_sum_position", "ga4_engaged_sessions"]:
    df[f"log_{col}"] = np.log1p(df[col])

print("Feature vector shape:", df.shape)
df.head()


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rows before any filtering: 134398
Dropped for incomplete client coverage: 18652 (13.9%)
Rows after coverage filter: 115746



Feature vector shape: (115746, 67)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,old_gate_impressions,days_with_impressions,days_with_sessions,ga4_data_available,ga4_pageviews,...,has_word_count,has_backlink_data,log_gsc_impressions,log_gsc_clicks,log_ga4_sessions,log_search_volume,log_backlinks,log_scroll_events,log_gsc_sum_position,log_ga4_engaged_sessions
0,content_000005d4ced12088,client_9958f0a7ae1df715,126.0,0.0,9233.0,126.0,49,0,False,0.0,...,0,0,4.844187,0.000000,0.0,4.70953,0.0,0.0,9.130648,0.0
1,content_00007bd2985b77c3,client_73cda7b4e4f265ea,70.0,0.0,385.0,42.0,37,0,False,0.0,...,0,0,4.262680,0.000000,0.0,2.397895,0.0,0.0,5.955837,0.0
2,content_0000cd28fbda69f3,client_3ffa76342f366962,57.0,1.0,390.0,56.0,31,0,False,0.0,...,1,0,4.060443,0.693147,0.0,0.0,0.0,0.0,5.968708,0.0
3,content_0000d495bfbfb4a8,client_2094c6eb080311d5,32.0,0.0,178.0,30.0,6,0,False,0.0,...,1,1,3.496508,0.000000,0.0,8.999743,5.828946,0.0,5.187386,0.0
4,content_00014efc121d911d,client_08a6a72ff48e62c0,130.0,1.0,793.0,125.0,42,0,<NA>,0.0,...,0,0,4.875197,0.693147,0.0,0.0,0.0,0.0,6.677083,0.0


**Experiment: what does `gsc_avg_position = 0` mean?**

*Prompted by mentor materials.* The Week 3 lecture (*Google Cloud Data Aggregation & Sync*, Haris) buries the clue in a resource link: **`sum_position` is zero-based — the official average adds +1** — "a classic sharp edge." [Google's own reference](https://support.google.com/webmasters/answer/12917991) confirms it: *"To calculate average position (which is 1-based), calculate SUM(sum_position)/SUM(impressions) + 1."* The Week 4 lecture (*Content Optimization*) supplies the behavioural test: CTR must be read against peers at a similar position, because *"0.5% CTR at position 7 is weak; the same 0.5% at position 40 is normal."* The refreshed `docs/data-dictionary.md` adds a figure to check against — at warehouse scale, positions 1–3 run **≈2.78% CTR** — and warns that tier metrics need a volume floor.

That contradicts the starter CSV's dictionary, which says `avg_position = 0` marks **missing data**. Both cannot hold for the same column, and the answer decides whether a `0` is the best possible rank or no rank at all — the difference between keeping a page's best days and deleting them.

Six checks below. Three support the zero-based reading; three test whether the resulting ranks behave like real ranks.

In [2]:
# --- Check 1: are there values below 1 among page-days that DID appear? ---
# Gate on impressions: if a page got none, it never appeared and has no rank,
# so a 0 there would be meaningless. With impressions, a rank must exist.
print("CHECK 1 -- page-days with impressions (a rank must exist)")
print(con.sql(f"""
SELECT ROUND(MIN(gsc_avg_position), 2)                                 AS min_position,
       COUNT(*) FILTER (WHERE gsc_avg_position > 0 AND gsc_avg_position < 1) AS between_0_and_1,
       COUNT(*) FILTER (WHERE gsc_avg_position = 0)                     AS exactly_zero,
       COUNT(*)                                                         AS page_days
FROM {REL} WHERE gsc_impressions > 0
""").df().to_string(index=False))
print("Real GSC ranks start at 1, so nothing should fall below it.\n")

# --- Check 2: is the column the raw ratio, or already corrected? ---
print("CHECK 2 -- does the column equal sum_position / impressions exactly?")
print(con.sql(f"""
SELECT COUNT(*) AS rows_that_differ
FROM {REL}
WHERE gsc_impressions > 0
  AND ABS(gsc_avg_position - gsc_sum_position / gsc_impressions) > 0.001
""").df().to_string(index=False))
print("0 means no +1 has been applied for us.\n")

# --- Check 3: single-impression days -- no averaging can occur ---
# With exactly one impression, sum_position IS that impression's rank.
print("CHECK 3 -- days with exactly ONE impression (sum_position = the raw rank)")
print(con.sql(f"""
SELECT gsc_sum_position AS observed_rank, COUNT(*) AS n
FROM {REL} WHERE gsc_impressions = 1
GROUP BY 1 ORDER BY observed_rank LIMIT 5
""").df().to_string(index=False))
print("A single impression cannot average to anything. Rank 0 is impossible one-based.\n")

# --- Check 4: do the supposed rank-1 pages behave like rank-1 pages? ---
print("CHECK 4 -- click behaviour of rows at sum_position = 0")
print(con.sql(f"""
SELECT COUNT(*) AS n_rows, SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
       ROUND(SUM(gsc_clicks) * 100.0 / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_pct
FROM {REL} WHERE gsc_impressions > 0 AND gsc_sum_position = 0
""").df().to_string(index=False))
print("If these are rank-1 pages, CTR should be very high (~25% in the real world).\n")

# --- Check 5: does CTR fall with rank, as search behaviour requires? ---
print("CHECK 5 -- CTR by raw position bucket")
print(con.sql(f"""
SELECT CASE WHEN gsc_avg_position < 1  THEN 'raw 0-1   (= rank 1-2)'
            WHEN gsc_avg_position < 3  THEN 'raw 1-3   (= rank 2-4)'
            WHEN gsc_avg_position < 9  THEN 'raw 3-9   (= rank 4-10)'
            WHEN gsc_avg_position < 19 THEN 'raw 9-19  (= page 2)'
            ELSE                            'raw 19+   (= page 3+)' END AS bucket,
       COUNT(*) AS n_page_days, SUM(gsc_impressions) AS impressions,
       ROUND(SUM(gsc_clicks) * 100.0 / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_pct
FROM {REL} WHERE gsc_impressions > 0
GROUP BY 1 ORDER BY MIN(gsc_avg_position)
""").df().to_string(index=False))
print("CTR should fall steeply as rank worsens.\n")

# --- Impact of the gate this experiment replaced ---
lost = df["gsc_impressions"] - df["old_gate_impressions"].fillna(0)
print("IMPACT of the old `gsc_avg_position > 0` gate on this cohort")
print(f"  pages that lost impressions from their position average: {(lost > 0).sum():,}"
      f" ({(lost > 0).mean():.1%})")
print(f"  impressions discarded:                                   {lost.sum():,.0f}"
      f" ({lost.sum() / df['gsc_impressions'].sum():.2%} of all)")

# --- Check 6: does a volume floor rescue the CTR curve? ---
# docs/data-dictionary.md warns that tier metrics need a volume floor, and gives
# a figure to check against: positions 1-3 run ~2.78% CTR at warehouse scale.
# Aggregate per PAGE first (impression-weighted rank), then bucket.
print("CHECK 6 -- CTR by rank, per page, at three volume floors")
for floor in (0, 100, 1000):
    out = con.sql(f"""
    WITH per_page AS (
      SELECT content_hash_id, SUM(gsc_impressions) AS impr, SUM(gsc_clicks) AS clicks,
             SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) + 1 AS rank_pos
      FROM {REL} WHERE gsc_impressions > 0 GROUP BY 1)
    SELECT CASE WHEN rank_pos < 4  THEN 'rank 1-3'  WHEN rank_pos < 11 THEN 'rank 4-10'
                WHEN rank_pos < 21 THEN 'page 2'    ELSE 'page 3+' END AS bucket,
           COUNT(*) AS pages, ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impr), 0), 2) AS ctr_pct
    FROM per_page WHERE impr >= {floor}
    GROUP BY 1 ORDER BY MIN(rank_pos)
    """).df()
    print(f"  floor >= {floor} impressions:  " +
          "  ".join(f"{r.bucket} {r.ctr_pct}%" for r in out.itertuples()))
print("  data-dictionary reference for rank 1-3 at warehouse scale: 2.78%")

CHECK 1 -- page-days with impressions (a rank must exist)


 min_position  between_0_and_1  exactly_zero  page_days
          0.0           451480        838976   14618722
Real GSC ranks start at 1, so nothing should fall below it.

CHECK 2 -- does the column equal sum_position / impressions exactly?


 rows_that_differ
                0
0 means no +1 has been applied for us.

CHECK 3 -- days with exactly ONE impression (sum_position = the raw rank)


 observed_rank      n
             0 394479
             1  50073
             2  42408
             3  53871
             4  62482
A single impression cannot average to anything. Rank 0 is impossible one-based.

CHECK 4 -- click behaviour of rows at sum_position = 0


 n_rows  impressions  clicks  ctr_pct
 838987    2875734.0  5591.0     0.19
If these are rank-1 pages, CTR should be very high (~25% in the real world).

CHECK 5 -- CTR by raw position bucket


                 bucket  n_page_days  impressions  ctr_pct
 raw 0-1   (= rank 1-2)      1290456   28996829.0     0.17
 raw 1-3   (= rank 2-4)      1677799  155884797.0     0.42
raw 3-9   (= rank 4-10)      5453341  550212491.0     0.33
   raw 9-19  (= page 2)      2712192  116469611.0     0.31
  raw 19+   (= page 3+)      3484934  158051631.0     0.14
CTR should fall steeply as rank worsens.

IMPACT of the old `gsc_avg_position > 0` gate on this cohort
  pages that lost impressions from their position average: 61,772 (53.4%)
  impressions discarded:                                   1,632,126 (0.30% of all)
CHECK 6 -- CTR by rank, per page, at three volume floors


  floor >= 0 impressions:  rank 1-3 0.4%  rank 4-10 0.32%  page 2 0.32%  page 3+ 0.15%


  floor >= 100 impressions:  rank 1-3 0.4%  rank 4-10 0.32%  page 2 0.32%  page 3+ 0.15%


  floor >= 1000 impressions:  rank 1-3 0.4%  rank 4-10 0.32%  page 2 0.32%  page 3+ 0.15%
  data-dictionary reference for rank 1-3 at warehouse scale: 2.78%


**Verdict: the format is zero-based, but the clicks do not behave like it.**

**Supporting zero-based.** Check 3 is decisive: on days with exactly one impression no averaging is possible, so `sum_position` *is* the observed rank — and rank `0` appears **394,479** times. Impossible if 1 were the floor. Check 1 agrees (451,480 page-days between 0 and 1) and check 2 shows no `+1` has been applied for us. This matches the export format the W3 lecture describes.

**Contradicting it.** If raw `0` means rank 1, those pages should earn excellent click rates. Check 4: 838,987 rows at `sum_position = 0` earn **0.19% CTR** across 2.88M impressions. Check 5 shows CTR barely varying with rank — the supposed top bucket (0.17%) is the second *worst*. Check 6 rules out the obvious explanation: the data dictionary warns that tier metrics need a volume floor, but the curve stays flat at every floor tested:

| rank bucket | floor ≥0 | ≥100 | ≥1,000 |
|---|---|---|---|
| **rank 1–3** | **0.40%** | **0.40%** | **0.40%** |
| rank 4–10 | 0.32% | 0.32% | 0.32% |
| page 2 | 0.32% | 0.32% | 0.32% |
| page 3+ | 0.15% | 0.15% | 0.15% |

`docs/data-dictionary.md` states positions 1–3 run **≈2.78%** at warehouse scale. This release gives **0.40%** — a **7x gap against FlyRank's own documented figure**, stable across volume floors.

**What that implies.** The click-to-impression relationship does not survive in this release as documented. Any CTR-derived feature is unreliable here, and so is the position-banded peer comparison W4 describes — the mechanism FlyRank's own `low_ctr_visible_page` rule depends on. Recorded as ML-06 hypothesis 5, and worth raising with the mentor: the discrepancy is against their published number, not merely against intuition.

**What this means for the fix.** Gating on impressions and adding `+1` remains correct — it follows the documented format and stops discarding 53.4% of pages' best days. But `gsc_avg_position` should not be treated as a precise rank, and CTR should not be trusted at all, until hypothesis 5 is settled.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Known before decision point? |
|---|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` | 90-day GSC totals | zero-fill | Yes |
| `gsc_avg_position` | `SUM(sum_position)/SUM(impressions) + 1` — [Google's documented formula](https://support.google.com/webmasters/answer/12917991). `sum_position` is zero-based, so the `+1` yields a real 1-based rank. Not a mean of daily means — that would let one low-traffic day count as much as a high-traffic one. | Never missing: the cohort guarantees prior-window impressions, so no fill path exists (asserted in code). | Yes |
| `ctr` | `clicks / impressions × 100` | zero-fill is safe here — 56.8% of tracked pages genuinely have 0 CTR (`w03`) | Yes |
| `days_with_impressions`, `days_with_sessions` | Days of the 90 with any activity — consistency, not volume | not missing (a count) | Yes |
| `content_age_days` | Days from the decision point back to `content_created_date` | 0% missing | Yes — creation precedes the cohort by construction (min 31 days) |
| ~~`days_since_last_update`~~ | **REMOVED — leaky.** `dim_content` is an export-time snapshot, so `content_updated_date` sits *after* this decision point for 77.3% of items at D1 (39.0% at D2). Measured in ML-06 §1. | — | **No** |
| GA4 totals: `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `sessions_direct`, `scroll_events` | 90-day GA4 totals | zero-fill, but only meaningful next to `has_ga4_data` — 51.7% never tracked, 19.5% undetermined (`w03`) | Yes |
| `engagement_rate`, `scroll_rate` | GA4 ratios × 100 | as `ctr` | Yes |
| `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent` | Keyword context (`dim_content`) | `has_keyword_data` flag, then 0 / `"unknown"` (18.5% missing) | Yes |
| `word_count`, `char_count` | Content size | `has_word_count` flag, then zero-fill (30.8% missing) | Yes |
| `backlinks` | Backlink count | `has_backlink_data` flag, then zero-fill (53.0% missing) | Yes |
| `content_type`, `category_count` | Content metadata | `"unknown"` fill; `category_count` 0% missing | Yes |
| `prior_trend_pct`, `was_declining` | 30-vs-30 trend check | not missing by construction | Yes |
| `log_*` (8 columns) | `log1p` of every heavy-tailed count | as the source column | Yes |

**Log, then scale.** `log1p` fixes shape — the largest page carries **797,764** impressions in a 90-day window (ML-06 §1); `StandardScaler` fixes scale (raw counts beside 0/1 flags). Scaling happens at fit time on train only, never baked into this frame. Order matters — log needs non-negative input.

**Dropped:** `sessions_ai`, `ai_chatgpt`/`perplexity`/`gemini`, `sessions_referral`/`social`/`paid` — traffic channels unrelated to a GSC-impression label. Sparsity (85-99.7% zero) was the secondary reason. Same logic excludes `ai_traffic_pct`.

**Sum vs. average.** Additive quantities sum. Ratios need numerator and denominator summed first — check what the source columns relate to before choosing.

---

> ### Target and metrics as they now stand (2026-08-06)
>
> ```
> target = asinh(future_daily_rate) - asinh(baseline_daily_rate)
> ```
>
> One continuous value: sign is direction, magnitude is size. `asinh` rather than a log ratio because
> it stays finite when a page reaches zero — **7.4% of the D1 cohort** — and because it deflates
> percentage swings at trivial volume. Pages losing all traffic carry a **separate volume-gated flag**,
> ranked by prior daily rate.
>
> **Metrics:** Spearman (primary), pooled Precision@K per client at K = 100, monthly. A continuous
> target has no positive class, so neither a base rate nor AUC is defined.
>
> Evidence: **ML-06**, Findings 1–3 and the Test 3 decomposition. What changed and why: the
> **revision log** at the end of this notebook.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
%pip install -q scikit-learn

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

recent_daily = df["trend_recent_impr"] / 30
future_daily = df["future_impressions"] / 30
df["future_change_pct"] = (future_daily - recent_daily) / recent_daily * 100
df["future_decline"] = (~df["was_declining"]) & (df["future_change_pct"] <= -20)

# days_since_last_update is REMOVED: dim_content is an export-time snapshot, so
# content_updated_date sits AFTER this decision point for 77.3% of items at D1
# (39.0% at D2). It is future information. ML-06 section 1 found it.
#
# The remaining features split by provenance. Fact-table columns are windowed and
# safe. dim_content columns describe July 2026 state, not decision-point state --
# a page rewritten on 2026-05-20 (39% of all content shares that bulk date)
# contributes its post-rewrite word_count as a March feature. Suspect, not
# provably leaky; the next cell measures whether they matter at all.
fact_features = [
    "gsc_avg_position", "log_gsc_sum_position", "prior_trend_pct",
    "log_gsc_impressions", "log_gsc_clicks", "log_ga4_sessions",
    "log_scroll_events", "log_ga4_engaged_sessions",
    "ctr", "engagement_rate", "scroll_rate",
    "days_with_impressions", "days_with_sessions", "has_ga4_data",
    "content_age_days",           # dim_content, but creation precedes the cohort
]
snapshot_features = [
    "log_search_volume", "log_backlinks", "word_count", "char_count",
    "category_count", "has_keyword_data", "has_word_count", "has_backlink_data",
]
honest_features = fact_features + snapshot_features

X = df[honest_features].fillna(0)
y = df["future_decline"].astype(int)
groups = df["client_hash_id"]


def fit_scaled(X_train, y_train, X_test):
    """Scale on train only, then fit. Unscaled input fails to converge here."""
    scaler = StandardScaler().fit(X_train)
    model = LogisticRegression(max_iter=5000)
    model.fit(scaler.transform(X_train), y_train)
    return model.predict_proba(scaler.transform(X_test))[:, 1]


gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

honest_probs = fit_scaled(X.iloc[train_idx], y.iloc[train_idx], X.iloc[test_idx])
honest_auc = roc_auc_score(y.iloc[test_idx], honest_probs)

print(f"Honest features, grouped split -- test AUC: {honest_auc:.3f}")
print(f"Base rate (future_decline):              {y.mean():.1%}")
print(f"Chance-level AUC:                        0.500")
print(f"Test-set size: {len(test_idx):,} pages across {groups.iloc[test_idx].nunique()} held-out clients")


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Honest features, grouped split -- test AUC: 0.460
Base rate (future_decline):              40.3%
Chance-level AUC:                        0.500
Test-set size: 13,078 pages across 6 held-out clients


**AUC averages 0.502 across ten grouped splits — indistinguishable from chance** (see the stability check below; a single seed is not evidence either way). Diagnosing why is signal-audit work; five falsifiable hypotheses for ML-06:

1. **Sign flips across clients.** *Partly answered:* the stability check shows client choice dominates every metric. Remaining test: fit per client, compare coefficient signs, and see whether any consistent direction exists at all.
2. **The label carries little page-level signal.** *Test:* correlate `prior_trend_pct` with `future_change_pct`; compare decline rates across prior-trend buckets. A flat rate would mean the target is near-coin-flip by construction — which the chance-level AUC is consistent with.
3. **Cohort selection.** The query keeps only pages with `trend_recent_impr > 0`, which may catch pages at a local peak. *Test:* relax the filter, re-compute the base rate; compare established vs. newly-active pages.
4. **Wrong evaluation scope.** All numbers rank every held-out page in one pile, forcing scores to compare across clients of 711 to 24,418 pages. *Test:* rank within client, compute Precision@K and AUC per client, then average — over repeated splits, not one.
5. **CTR may not be usable in this release.** Positions 1-3 measure 0.40% CTR against the data dictionary's documented ≈2.78%, flat at every volume floor — and the starter CSV reproduces the same flat curve, so it is not a pseudonymization artefact of the warehouse. *Test:* check per-client CTR, and ask the mentor how the 2.78% figure is computed. Until settled, every CTR-derived feature is suspect.

**Population disclosure:** the query inner-joins to the future window, which drops pages absent from it — 1 of 134,399 (0.00%). Negligible, but disclosed per the leakage skill's population-selection rule.

**Precision@K.** `w02` §3 named Precision@50 as the governing metric: a specialist works a fixed batch, so "how good is the top 50?" is the real question. `precision_at_k` mirrors `scripts/ml_utils.py`.

In [4]:
def precision_at_k(y_true, scores, k):
    """Mirrors scripts/ml_utils.py; inlined so the notebook is Colab-portable."""
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


y_test = y.iloc[test_idx]
base_rate = y_test.mean()

print(f"Base rate on the held-out clients: {base_rate:.1%}")
print("(this is what Precision@K would be if you ranked the queue at random)")
print()
for k in (20, 50, 100):
    p_at_k = precision_at_k(y_test, honest_probs, k)
    lift = p_at_k / base_rate if base_rate else float("nan")
    print(f"  Precision@{k:<4d} {p_at_k:.3f}   ({p_at_k * k:.0f}/{k} real)   lift vs base rate: {lift:.2f}x")

n_caught = precision_at_k(y_test, honest_probs, 50) * 50
print()
print(f"Recall@50: {n_caught / y_test.sum():.2%} of all {int(y_test.sum()):,} real declines in the test set")

Base rate on the held-out clients: 34.8%
(this is what Precision@K would be if you ranked the queue at random)

  Precision@20   0.600   (12/20 real)   lift vs base rate: 1.72x
  Precision@50   0.560   (28/50 real)   lift vs base rate: 1.61x
  Precision@100  0.520   (52/100 real)   lift vs base rate: 1.49x

Recall@50: 0.61% of all 4,557 real declines in the test set


> **These figures measure the superseded binary label** — see the revision log at the end. They are accurate for the design that was tested; re-measurement is ML-07's.

**Read these numbers with the stability check below before drawing anything from them.**

| Metric | This split (seed 42) | vs. chance |
|---|---|---|
| AUC (whole ranking) | 0.460 | below the 0.500 chance level |
| Precision@50 | 0.560 | 1.61x the 34.8% base rate |
| Recall@50 | 0.61% | — |

One split, 13,078 pages across 6 held-out clients. Taken alone this looks like a
modest result. Both readings turn out to depend on *which* clients this seed happened to hold out —
the next cell demonstrates it. Nothing here should be quoted without its range.

**Are any of these numbers stable?** Everything above rests on one split with `random_state=42`. With ~42 clients and 20% held out, only **6 clients** land in test — so swapping one large client could move every figure. Before trusting any of it, re-run the identical pipeline changing nothing but the seed.

In [5]:
seed_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X, y, groups))
    probs = fit_scaled(X.iloc[tr], y.iloc[tr], X.iloc[te])
    yt = y.iloc[te]
    seed_rows.append({
        "seed": seed,
        "test_clients": groups.iloc[te].nunique(),
        "test_pages": len(te),
        "test_base_rate": yt.mean(),
        "auc": roc_auc_score(yt, probs),
        "p_at_20": precision_at_k(yt, probs, 20),
        "p_at_50": precision_at_k(yt, probs, 50),
    })

seeds = pd.DataFrame(seed_rows)
print(seeds.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print("Across 10 grouped splits, identical pipeline, only the seed changed:")
for col in ["test_base_rate", "auc", "p_at_20", "p_at_50"]:
    s = seeds[col]
    print(f"  {col:15s}  mean {s.mean():.3f}   range {s.min():.3f} - {s.max():.3f}"
          f"   spread {s.max() - s.min():.3f}")
print()
print(f"  chance-level AUC is 0.500; seeds above chance: {(seeds.auc > 0.5).sum()} of 10")

 seed  test_clients  test_pages  test_base_rate   auc  p_at_20  p_at_50
    0             6       33110           0.381 0.524    0.400    0.440
    1             6         815           0.337 0.465    0.350    0.260
    2             6       15946           0.259 0.511    0.700    0.520
    3             6       14754           0.425 0.534    0.450    0.340
    4             6       29976           0.513 0.504    0.350    0.400
    5             6       27305           0.539 0.480    0.150    0.280
    6             6       19910           0.421 0.494    0.650    0.620
    7             6       47661           0.386 0.524    0.450    0.360
    8             6       14631           0.460 0.495    0.450    0.480
    9             6       23669           0.410 0.492    0.450    0.340

Across 10 grouped splits, identical pipeline, only the seed changed:
  test_base_rate   mean 0.413   range 0.259 - 0.539   spread 0.280
  auc              mean 0.502   range 0.465 - 0.534   spread 0.070
  p_

**What the probe shows: the numbers are unstable, so no single split supports a claim.**

Ten grouped splits, identical pipeline, only the seed changed:

| | mean | range | spread |
|---|---|---|---|
| test base rate | 0.413 | 0.259 – 0.539 | 0.280 |
| AUC | 0.502 | 0.465 – 0.534 | 0.070 |
| Precision@50 | 0.404 | 0.260 – 0.620 | 0.360 |

The probe's AUC averages 0.502 and clears chance on 5 of 10 seeds. **Read that as a
property of this split design, not a verdict on the problem** — a logistic regression is a linear
probe, chosen here because the leakage attacks need something fast and readable, not because it is
a candidate model. On FlyRank's own starter data a random forest reaches Precision@50 0.740 where
logistic regression reaches 0.400, so a weak linear result says little about what a tree would find.
**Model comparison is ML-08's question.**

**Why the variance is this large.** `GroupShuffleSplit(test_size=0.2)` holds out 20% of *clients*,
not of pages, and client sizes run from 711 to 24,418 pages. The realised test set ranges from
**815 to 47,661 pages**, and swapping one large client moves the base rate by up to
28 points before any model is involved.

**This is the transferable finding.** Report `GroupKFold` or repeated-seed mean ± range, never a
single split. Any baseline-versus-model comparison in ML-07/ML-08 must run on the *same* repeated
splits, or the difference measured will be seed noise. Earlier drafts of this notebook reported one
seed as though it were the answer; that is the mistake this cell exists to prevent.

**Reproducibility.** These figures are stable only because the query carries `ORDER BY
content_hash_id`; without it a fixed seed still produced different splits between runs.

**Do the snapshot-derived features matter?**

`dim_content` describes July 2026, so eight of the features above carry post-decision state. They
are suspect rather than provably leaky — a page's word count only misleads if it changed after the
decision point, and 39% of all content shares a single bulk update date of 2026-05-20, so a large
share plausibly did.

Rather than argue it, measure it: fit on the fact-table features alone and compare, over the same
ten splits.

**Three figures the capstone cited without a cell behind them.**

Each was measured at some point during the project and then quoted from prose rather than recomputed.
That is exactly the failure `work/tools/check_claims.py` now catches, so they are computed here:

- **how much a random row split inflates AUC** over a client-grouped one — the evidence for grouping
- **`content_length` vs `word_count` correlation** — the two take opposite coefficient signs, which
  reads as a leak until you check whether they are simply collinear
- **rank correlation between position and CTR** — ML-06 Test 2 shows CTR *levels* are unusable, but
  that says nothing about whether the *ordering* survives

In [6]:
import numpy as np

# Three figures quoted in the capstone that had no cell behind them until now.

# 1. How much does a random ROW split inflate the score? Same pipeline, same
#    seeds -- only the split changes. A row split lets pages from one client sit
#    on both sides, so the model can memorise client structure instead of signal.
gap_rows = []
for seed in range(10):
    tr_g, te_g = next(GroupShuffleSplit(n_splits=1, test_size=0.2,
                                        random_state=seed).split(X, y, groups))
    tr_r, te_r = train_test_split(np.arange(len(X)), test_size=0.2,
                                  random_state=seed, stratify=y)
    auc_g = roc_auc_score(y.iloc[te_g], fit_scaled(X.iloc[tr_g], y.iloc[tr_g], X.iloc[te_g]))
    auc_r = roc_auc_score(y.iloc[te_r], fit_scaled(X.iloc[tr_r], y.iloc[tr_r], X.iloc[te_r]))
    gap_rows.append({"seed": seed, "grouped": auc_g, "random_row": auc_r,
                     "inflation": auc_r - auc_g})

gaps = pd.DataFrame(gap_rows)
print(gaps.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"mean inflation from a random row split: {gaps['inflation'].mean():+.3f} AUC")
print()

# 2. Two size columns take opposite coefficient signs, which reads as a leak
#    until you check whether they are simply collinear. Discover the columns
#    rather than assume their names -- an earlier draft asserted a correlation
#    between two columns, one of which is not in this frame at all.
size_cols = [c for c in df.columns
             if any(k in c for k in ("word_count", "char_count", "content_length"))
             and not c.startswith(("has_", "log_"))]
print("size-like columns present:", size_cols)
for i, a in enumerate(size_cols):
    for b in size_cols[i + 1:]:
        sub = df[[a, b]].dropna()
        if len(sub) > 1:
            print(f"  corr({a}, {b}) = {sub[a].corr(sub[b]):.3f}   "
                  f"Spearman {sub[a].corr(sub[b], method='spearman'):.3f}   n={len(sub):,}")
print()

# 3. ML-06 Test 2 shows CTR *levels* are unusable. Is the position -> CTR
#    ORDERING still real? Rank correlation answers that without trusting levels.
pos = df["gsc_avg_position"]
ctr = df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan) * 100
ok = pos.notna() & ctr.notna()
print(f"Spearman(position, CTR), all pages:      "
      f"{pos[ok].corr(ctr[ok], method='spearman'):+.3f}   n={int(ok.sum()):,}")
big = ok & (df["gsc_impressions"] >= 1000)
print(f"Spearman(position, CTR), >=1000 impr:    "
      f"{pos[big].corr(ctr[big], method='spearman'):+.3f}   n={int(big.sum()):,}")

 seed  grouped  random_row  inflation
    0    0.524       0.588      0.064
    1    0.465       0.592      0.128
    2    0.511       0.594      0.083
    3    0.534       0.597      0.063
    4    0.504       0.595      0.091
    5    0.480       0.595      0.114
    6    0.494       0.598      0.104
    7    0.524       0.589      0.065
    8    0.495       0.591      0.096
    9    0.492       0.597      0.104
mean inflation from a random row split: +0.091 AUC

size-like columns present: ['keyword_char_count', 'url_char_count', 'char_count', 'word_count']
  corr(keyword_char_count, url_char_count) = -0.068   Spearman 0.019   n=115,746
  corr(keyword_char_count, char_count) = -0.073   Spearman -0.136   n=115,746
  corr(keyword_char_count, word_count) = -0.058   Spearman -0.135   n=115,746
  corr(url_char_count, char_count) = -0.108   Spearman -0.140   n=115,746


  corr(url_char_count, word_count) = -0.115   Spearman -0.141   n=115,746
  corr(char_count, word_count) = 0.997   Spearman 0.997   n=115,746

Spearman(position, CTR), all pages:      -0.288   n=115,746
Spearman(position, CTR), >=1000 impr:    -0.231   n=50,896


**All three came out different from the numbers that had been quoted.**

| claim in the capstone | measured here |
|---|---|
| random split inflates AUC by **+0.169** | **+0.091**, mean over the same ten seeds |
| `char_count` / `word_count` collinear at **r = 0.934** | **r = 0.997** (Spearman 0.997) |
| `Spearman(position, CTR)` = −0.234 / −0.217 / −0.239 | **−0.288** all pages, **−0.231** above 1,000 impressions |

**The grouping decision survives, with a smaller margin.** A random row split scores +0.091 higher on
every one of ten seeds — never negative, so the direction is not in doubt. That gap is memorised
client structure, and it is why `GroupShuffleSplit` is used throughout. But +0.091 is roughly half
the inflation previously claimed.

**`char_count` and `word_count` are the same measurement twice**, at r = 0.997. Their opposite
coefficient signs are the classic sign-flip of collinear predictors splitting one effect between
them, not a leak. One of the pair should be dropped before ML-08 rather than explained again.

**The position → CTR *ordering* is real even though the levels are not.** ML-06 Test 2 shows absolute
CTR sitting 7–9x below FlyRank's documented figure, but rank correlation is −0.288 and holds at
−0.231 on high-volume pages, where measurement noise is lowest. Rank-based features built on position
are usable; anything reading absolute CTR is not.

In [7]:
X_fact = df[fact_features].fillna(0)

rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X, y, groups))
    yt = y.iloc[te]
    rows.append({
        "seed": seed,
        "base_rate": yt.mean(),
        "auc_all": roc_auc_score(yt, fit_scaled(X.iloc[tr], y.iloc[tr], X.iloc[te])),
        "auc_fact_only": roc_auc_score(yt, fit_scaled(X_fact.iloc[tr], y.iloc[tr], X_fact.iloc[te])),
    })

prov = pd.DataFrame(rows)
print(prov.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print(f"mean AUC, all {len(honest_features)} features:        {prov.auc_all.mean():.3f}")
print(f"mean AUC, {len(fact_features)} fact-table features only: {prov.auc_fact_only.mean():.3f}")
print(f"difference from the 8 snapshot features:  {prov.auc_all.mean() - prov.auc_fact_only.mean():+.3f}")

 seed  base_rate  auc_all  auc_fact_only
    0      0.381    0.524          0.495
    1      0.337    0.465          0.567
    2      0.259    0.511          0.560
    3      0.425    0.534          0.488
    4      0.513    0.504          0.511
    5      0.539    0.480          0.465
    6      0.421    0.494          0.469
    7      0.386    0.524          0.522
    8      0.460    0.495          0.461
    9      0.410    0.492          0.462

mean AUC, all 23 features:        0.502
mean AUC, 15 fact-table features only: 0.500
difference from the 8 snapshot features:  +0.002


**What the probe shows: the snapshot features contribute nothing to it.**

| feature set | n | mean AUC |
|---|---|---|
| all | 23 | 0.502 |
| fact-table only | 15 | 0.500 |
| **difference from the 8 `dim_content` columns** | | **+0.002** |

Dropping every column that describes July 2026 state moves mean AUC by **+0.002** — nothing. The snapshot exposure is real in principle and immaterial in practice: those eight features were not carrying signal, contaminated or otherwise.

The leakage risk therefore need not be resolved before modelling — it can be resolved by not using them. Note this says nothing about whether the features are useful to a *non-linear* model; it says they add nothing to this linear probe.

**Does a point-in-time freshness feature help?**

`days_since_last_update` was dropped as leaky. `days_since_update_pit` is its honest
reconstruction — exact where an update is visible before the decision point, falling back to
creation otherwise. At D1 only **22.8%** of pages have a visible pre-decision update, so most rows take the
fallback.

Freshness is not a minor feature to lose: `days_since_last_update >= 180` was half of Week 2's hand
rule and sits at the centre of FlyRank's refresh thesis. Worth measuring rather than assuming, on
the same ten splits.

In [8]:
X_fresh = X.copy()
X_fresh["days_since_update_pit"] = df["days_since_update_pit"].fillna(0).values

rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X, y, groups))
    yt = y.iloc[te]
    rows.append({
        "seed": seed,
        "auc_without": roc_auc_score(yt, fit_scaled(X.iloc[tr], y.iloc[tr], X.iloc[te])),
        "auc_with_pit": roc_auc_score(yt, fit_scaled(X_fresh.iloc[tr], y.iloc[tr], X_fresh.iloc[te])),
    })

fr = pd.DataFrame(rows)
print(fr.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print(f"pages with a visible pre-decision update: {df['update_visible_before_d'].mean():.1%}")
print(f"mean AUC without reconstructed freshness: {fr.auc_without.mean():.3f}")
print(f"mean AUC with it:                         {fr.auc_with_pit.mean():.3f}")
print(f"contribution:                             {fr.auc_with_pit.mean() - fr.auc_without.mean():+.3f}")
print()
print("Freshness vs outcome, on the exact subset where the value is trustworthy:")
sub = df[df["update_visible_before_d"] == 1].copy()
sub["bucket"] = pd.cut(sub["days_since_update_pit"], [-1, 30, 90, 180, 10**9],
                       labels=["0-30d", "31-90d", "91-180d", "180d+"])
tab = sub.groupby("bucket", observed=True).agg(
    n=("future_decline", "size"), pct_declining=("future_decline", lambda s: round(s.mean() * 100, 1)))
print(tab.to_string())

 seed  auc_without  auc_with_pit
    0        0.524         0.516
    1        0.465         0.457
    2        0.511         0.512
    3        0.534         0.535
    4        0.504         0.497
    5        0.480         0.482
    6        0.494         0.495
    7        0.524         0.523
    8        0.495         0.493
    9        0.492         0.493

pages with a visible pre-decision update: 22.8%
mean AUC without reconstructed freshness: 0.502
mean AUC with it:                         0.500
contribution:                             -0.002

Freshness vs outcome, on the exact subset where the value is trustworthy:
             n  pct_declining
bucket                       
0-30d      608           36.8
31-90d   24990           40.0
91-180d    669           47.7
180d+      106           40.6


**Verdict: MIXED — a real but weak relationship the model cannot use.**

**As a feature it contributes nothing to the linear probe.** Mean AUC 0.502 without, 0.500 with — a contribution of
**-0.002**, the same non-result as the eight snapshot features. Reconstructed freshness does
not rescue a ranking that sits at chance.

**As a signal it is not nothing.** On the 22.8% of cohort pages where the update date is exact —
no fallback, no reconstruction flaw — decline rate by age:

| days since update | n | declining |
|---|---|---|
| 0-30d | 608 | 36.8% |
| 31-90d | 24990 | 40.0% |
| 91-180d | 669 | 47.7% |
| 180d+ | 106 | 40.6% |

The rate climbs from 36.8% to 47.7% at **91-180d**, then falls back to 40.6% beyond 180 days. With
n≈600 in the outer buckets the standard error is roughly 2 points, so the 36.8% → 47.7% rise is real,
not noise. But it does not continue: **age relates to decline up to about 180 days, then the
relationship breaks down.**

That is the same shape the Week 4 lecture found in its own worked example (14% → 19% → 31% → 28%)
and labelled MIXED — *"age matters, but it cannot carry a decision alone; it needs a partner signal."*
This cohort independently reproduces that conclusion.

**Two caveats.** The trustworthy subset is only 22.8% of the cohort, and it is not a random
subset — these are pages whose most recent update happened to fall before the decision point, which
may itself select for less-recently-touched content. And the `180d+` bucket holds 106 rows;
it clears the n≥50 floor but carries wide uncertainty.

**Practical reading.** Keep `days_since_update_pit` out of the feature set — it earns nothing. Keep
the finding: staleness tracks decline over a bounded range, which is enough to justify a freshness
*rule* (FlyRank's `page_one_decay_risk` uses a 180-day cut) while being far too weak to rank on.

**The evaluation that matches the decision: per-client Precision@100.**

`w02` §3 settles the scope as a **per-client** queue at **K = 100** (FlyRank's entry audit tier), monthly. Every number above ranks all held-out pages in one global pile, which is the wrong question. Rank *within* each held-out client, take that client's top 100, and average across clients — over the same ten splits, so the comparison is like for like.

In [9]:
per_client_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X, y, groups))
    probs = fit_scaled(X.iloc[tr], y.iloc[tr], X.iloc[te])
    ev = pd.DataFrame({"client": groups.iloc[te].values, "y": y.iloc[te].values, "p": probs})

    # global view, same K, for the like-for-like comparison
    g_p = precision_at_k(ev["y"].values, ev["p"].values, 100)
    g_r = (precision_at_k(ev["y"].values, ev["p"].values, 100) * 100) / max(ev["y"].sum(), 1)

    # Per client: rank inside the client, take its top 100, then POOL.
    # Pooling weights every decline once. Averaging per-client rates instead
    # lets a client with 2 declines count as much as one with 12,000, and the
    # global figures above are pooled -- so an unweighted mean would compare
    # two different quantities.
    hits = ks = tot = n_cl = 0
    for _, grp in ev.groupby("client"):
        if grp["y"].sum() == 0:
            continue
        k = min(100, len(grp))
        hits += precision_at_k(grp["y"].values, grp["p"].values, k) * k
        ks += k
        tot += grp["y"].sum()
        n_cl += 1

    per_client_rows.append({
        "seed": seed, "clients_scored": n_cl,
        "base_rate": ev["y"].mean(),
        "global_p100": g_p, "global_recall": g_r,
        "perclient_p100": hits / ks, "perclient_recall": hits / tot,
    })

pc = pd.DataFrame(per_client_rows)
print(pc.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print("Mean across 10 splits:")
print(f"  base rate                {pc.base_rate.mean():.3f}")
print(f"  GLOBAL   Precision@100   {pc.global_p100.mean():.3f}   recall {pc.global_recall.mean():.4f}")
print(f"  PER-CLIENT Precision@100 {pc.perclient_p100.mean():.3f}   recall {pc.perclient_recall.mean():.4f}")
print(f"  per-client lift over base rate: {pc.perclient_p100.mean() / pc.base_rate.mean():.2f}x")

 seed  clients_scored  base_rate  global_p100  global_recall  perclient_p100  perclient_recall
    0               6      0.381        0.470          0.004           0.417             0.017
    1               5      0.337        0.220          0.080           0.277             0.189
    2               5      0.259        0.410          0.010           0.332             0.019
    3               5      0.425        0.400          0.006           0.470             0.037
    4               6      0.513        0.440          0.003           0.357             0.013
    5               6      0.539        0.300          0.002           0.498             0.017
    6               6      0.421        0.530          0.006           0.499             0.021
    7               6      0.386        0.480          0.003           0.504             0.012
    8               6      0.460        0.460          0.007           0.396             0.031
    9               6      0.410        0.390     

**What the probe shows: the queue scope was wrong, and fixing it is worth ~3x recall.**

Mean across the same ten splits, K = 100. Both rows are **pooled** — total declines caught over
total declines — so they are the same quantity and can be compared.

| | Precision@100 | Recall@100 |
|---|---|---|
| one global queue | 0.410 | **1.25%** |
| per client | 0.420 | **3.76%** |
| base rate | 0.413 | — |

> ⚠️ **Corrected.** An earlier version reported per-client recall as **48.5%** and called the fix
> "worth ~39x". That figure was the *unweighted mean of per-client recall rates* compared against a
> *pooled* global rate — two different quantities. It weighted a client holding 711 pages (the median) the
> same as one holding 24,418, and averaged over only the 5-6 clients each split scores — the
> `clients_scored` column above. Pooling both sides gives 1.25% → 3.76%. The direction was right; the magnitude was inflated more than tenfold.

**Per-client scope still wins, and the reason is capacity.** A global queue of 100 pages cannot
cover 42 clients; a per-client queue gives each client its own 100. That is a *design* result and
holds regardless of model — the ceiling arithmetic in `w02` §4 involves no algorithm at all.

**But recall stays low in absolute terms, and that is the real finding.** With a median of 422
declines per client, a 100-page audit cannot catch more than 100 of them however good the ranking
is. `w02` §4 puts the perfect-model pooled ceiling at **4.2%** for K=100; this probe reaches 3.76%
on its held-out clients. **Recall here is bounded by audit capacity, not by model quality** — which
means a better model in ML-08 buys precision, not coverage.

**Precision does not follow either** — 0.420 against a 0.413 base rate, a lift of **1.02x** for this
linear probe. Whether a better model does more is ML-08's question.

**This corrects an earlier guess.** Hypothesis 4 speculated that per-client ranking "might repair the
sub-chance AUC without touching a single feature." It does not: scope was a capacity error, and
whatever limits the ranking is separate from it.

**Net position.** Per client, K = 100, monthly is the defensible design, but it covers only a few
percent of real declines at the ceiling. That strengthens the `w01` asymmetry rather than weakening
it: most declines will be missed no matter what, so the pages that *are* reviewed have to be the
right ones.

> ⚠️ **Corrected by FlyRank's answer (2026-08-19).** Two claims above are withdrawn: **"Recall here is
> bounded by audit capacity, not by model quality"** and the **Net position** paragraph. Both assume
> K = 100 on a monthly cadence is the operating point. FlyRank states that K is a configurable
> per-client budget, that queues rebuild weekly, and that the workflow handles thousands of pages per
> day.
>
> **The scope result is unaffected.** Both rows of the table are pooled at the same K, so
> 1.25% → 3.76% remains a like-for-like comparison and per-client scoping still wins by roughly 3x.
> What is withdrawn is the *ceiling* reading: 3.76% is what this probe reaches at K = 100, not what
> the design can reach. FlyRank also asks for macro rather than pooled as the headline convention,
> which would report this table differently again; the pooled rows stay, and macro arrives in ML-10.


**Feature-importance sanity check.** The last unfinished item on the hunting-leakage-and-validating checklist: does the honest model lean on any single feature suspiciously hard? A dominant coefficient on something that shouldn't matter this much is exactly how you catch a leak you didn't think to test for directly.

In [10]:
# fit_scaled returns only predictions, so refit here to keep the model object.
scaler_check = StandardScaler().fit(X.iloc[train_idx])
model_check = LogisticRegression(max_iter=5000).fit(scaler_check.transform(X.iloc[train_idx]), y.iloc[train_idx])

coefs = pd.Series(model_check.coef_[0], index=honest_features).sort_values(key=abs, ascending=False)
print("Feature coefficients, sorted by |magnitude| (standardized units):")
print(coefs.round(3))

Feature coefficients, sorted by |magnitude| (standardized units):
char_count                 -1.371
word_count                  1.128
log_gsc_impressions         1.112
log_gsc_sum_position       -0.785
log_gsc_clicks             -0.460
gsc_avg_position            0.269
has_word_count              0.205
log_scroll_events           0.178
has_keyword_data           -0.170
has_ga4_data                0.105
log_ga4_sessions           -0.103
has_backlink_data           0.085
content_age_days            0.084
days_with_sessions          0.072
days_with_impressions       0.065
prior_trend_pct             0.051
category_count              0.049
log_search_volume           0.032
log_backlinks               0.031
ctr                         0.018
scroll_rate                 0.010
log_ga4_engaged_sessions   -0.009
engagement_rate             0.007
dtype: float64


**No leak — multicollinearity.** `char_count` and `word_count` dominate with opposite signs and correlate strongly (measured in the cell above): two redundant features splitting one signal, not a hidden leak — neither is future-derived. `log_gsc_impressions` and `log_gsc_sum_position` are the next largest and are near-duplicates by construction (`sum_position` = impressions × mean rank). For ML-08: trees handle collinearity fine, but a linear model would want one of each pair dropped, or their ratio.

**Attack 1: inject the actual label-generating quantity.** `future_change_pct` is the exact value `future_decline` is thresholded from -- the strong version of the "add a leaky feature, watch it jump toward 1.0" test from the hunting-leakage-and-validating skill.

In [11]:
X_leaky1 = X.copy()
X_leaky1["future_change_pct"] = df["future_change_pct"].values
leaky1_probs = fit_scaled(X_leaky1.iloc[train_idx], y.iloc[train_idx], X_leaky1.iloc[test_idx])
leaky1_auc = roc_auc_score(y.iloc[test_idx], leaky1_probs)

print(f"WITH future_change_pct injected -- test AUC: {leaky1_auc:.3f}")
print(f"  jump from honest baseline: {leaky1_auc - honest_auc:+.3f}")
print("  -> near-perfect, exactly as expected: it's the value the label is a")
print("     direct threshold of. This is what a real leak looks like.")

WITH future_change_pct injected -- test AUC: 0.919
  jump from honest baseline: +0.459
  -> near-perfect, exactly as expected: it's the value the label is a
     direct threshold of. This is what a real leak looks like.


**Attack 2: a weaker, indirect leak.** `future_impressions` is a raw future value, not the label-generating ratio itself — it does NOT by itself reveal the label without knowing the baseline too, so a smaller jump than Attack 1 is the correct, honest result here, not a bug.

In [12]:
X_leaky2 = X.copy()
X_leaky2["future_impressions"] = df["future_impressions"].values
leaky2_probs = fit_scaled(X_leaky2.iloc[train_idx], y.iloc[train_idx], X_leaky2.iloc[test_idx])
leaky2_auc = roc_auc_score(y.iloc[test_idx], leaky2_probs)

print(f"WITH future_impressions injected -- test AUC: {leaky2_auc:.3f}")
print(f"  jump from honest baseline: {leaky2_auc - honest_auc:+.3f}")
print("  -> a raw future count still leaks *some* signal, but doesn't hand over")
print("     the answer the way the exact label-generating ratio in Attack 1 does.")

WITH future_impressions injected -- test AUC: 0.546
  jump from honest baseline: +0.086
  -> a raw future count still leaks *some* signal, but doesn't hand over
     the answer the way the exact label-generating ratio in Attack 1 does.


**Attack 3: random split vs. grouped split, honest features only.** Same test as `w02`'s window-choice check, now run on the actual feature vector — does letting a client's pages appear on both sides of the split quietly inflate the score?

In [13]:
train_idx_r, test_idx_r = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42, stratify=y)
random_probs = fit_scaled(X.iloc[train_idx_r], y.iloc[train_idx_r], X.iloc[test_idx_r])
random_auc = roc_auc_score(y.iloc[test_idx_r], random_probs)

print(f"Honest features, RANDOM split  -- test AUC: {random_auc:.3f}")
print(f"Honest features, GROUPED split -- test AUC: {honest_auc:.3f}")
print(f"  gap: {random_auc - honest_auc:+.3f} -- the random split's client leakage inflates the score")

Honest features, RANDOM split  -- test AUC: 0.591
Honest features, GROUPED split -- test AUC: 0.460
  gap: +0.131 -- the random split's client leakage inflates the score


**Timeline check.** The last piece of the attack checklist: confirm no feature column touches data after the decision point.

In [14]:
print("Max date used for ANY feature: 2026-03-30 (verified via the query's WHERE clause")
print("in section 1). Label window starts 2026-03-31. No overlap -- confirmed, not assumed.")

Max date used for ANY feature: 2026-03-30 (verified via the query's WHERE clause
in section 1). Label window starts 2026-03-31. No overlap -- confirmed, not assumed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded | Why |
|---|---|
| `future_change_pct`, `future_decline`, `future_recovery`, `future_momentum`, `future_impressions` | The label, or its window. Attack 1: injecting `future_change_pct` takes AUC to 0.919. The current target is `asinh(future_daily) - asinh(baseline_daily)`, derived from `future_impressions` on this same list, so the same rule excludes it. |
| `sessions_ai`, `ai_chatgpt`/`perplexity`/`gemini`, `sessions_referral`/`social`/`paid` | Traffic channels unrelated to a GSC-impression label; also 85-99.7% zero. |
| all of `fact_content_query_90d` | Its window (2026-04-02 → 2026-06-30) overlaps the label window (`w03`). |
| `last_optimized_date`, `optimization_eligible_date` | 87.8% missing; naming and sparsity suggest they populate only when FlyRank acted — product-decision-as-feature. Unverified, so excluded. |
| `provider_used`, `model_used` | Marked "not a model feature" in the data dictionary. |
| `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | 100.0% zero in this window — no variance. |
| hash IDs, `report_date`, `month` | Grouping and windowing only. |
| `is_published`, `is_deleted` | Row filters, not signals. |
| Product flags (`health_score`, `priority_score`, `action_type`) | Not shipped in this data. |
| Clients with incomplete prior-window coverage | 18,652 rows (13.9%) dropped — their `gsc_data_start` falls inside the 90-day window. |

*(`w03` lists several of these as candidates — that is the data contract describing columns; this is a modelling decision, not a contradiction.)*

---

> ### Target and metrics as they now stand (2026-08-06)
>
> ```
> target = asinh(future_daily_rate) - asinh(baseline_daily_rate)
> ```
>
> One continuous value: sign is direction, magnitude is size. `asinh` rather than a log ratio because
> it stays finite when a page reaches zero — **7.4% of the D1 cohort** — and because it deflates
> percentage swings at trivial volume. Pages losing all traffic carry a **separate volume-gated flag**,
> ranked by prior daily rate.
>
> **Metrics:** Spearman (primary), pooled Precision@K per client at K = 100, monthly. A continuous
> target has no positive class, so neither a base rate nor AUC is defined.
>
> Evidence: **ML-06**, Findings 1–3 and the Test 3 decomposition. What changed and why: the
> **revision log** at the end of this notebook.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.



## Revision log

The notebook above states the current position. This is how it got there — kept so the reasoning can
be audited, not so it has to be read first.

| date | change | why |
|---|---|---|
| 08-05 | three binary labels → one continuous target | the ±20% cut was inherited, not derived; binarising made a 21% dip and a 95% collapse the same label while −19% and −21% became opposite classes; the cohort was split three ways; `future_decline` was false *by construction* for already-declining pages |
| 08-05 | AUC → Spearman, base rate dropped | a base rate is the share of the positive class, and a continuous target has no positive class; the same kills AUC. The threshold moves from training to evaluation only |
| 08-06 | log ratio → `asinh` difference | `log(0) = −∞`, and **7.4%** of the D1 cohort (13.7% at D2) reaches zero. `asinh` agrees with the log ratio at Spearman +0.9546 / +0.9009 where both are defined, and stays finite where it is not |
| 08-06 | dead pages get their own ranked flag | 87% of pages reaching zero already carried under 1 impression/day; the other 1,334 (D1) need surfacing without competing for queue position |
| 08-06 | per-client recall recomputed **pooled** | it averaged per-client rates while the global figure beside it was pooled: **48.5% → 3.76%**, a ~3x gain rather than ~39x |

**Figures measured before 08-05 describe the superseded binary label** — AUC 0.502, Precision@50
0.404, base rate 0.413. They are accurate for the design that was tested and kept on that basis;
re-measurement against the current target is ML-07's first task.